<a href="https://colab.research.google.com/github/mkvkanpur/hpc_book/blob/main/warp_tile.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
try:
    import warp as wp
    print(f"Warp version {wp.__version__} is ready!")
except ImportError:
    print("Warp not found. Installing...")
    !pip install warp-lang
    import warp as wp

wp.init()
device = "cuda"

Warp not found. Installing...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 136.5/136.5 MB 7.6 MB/s eta 0:00:00
Warp 1.12.0 initialized:
   CUDA Toolkit 12.9, Driver 13.0
   Devices:
     "cpu"      : "x86_64"
     "cuda:0"   : "Tesla T4" (15 GiB, sm_75, mempool enabled)
   Kernel cache:
     /root/.cache/warp/1.12.0


In [ ]:
@wp.kernel
def compute(out_arr: wp.array(dtype=float)):

    a = wp.tile_arange(0.0, 1.0, 0.1, dtype=float)
    b = wp.tile_ones(shape=10, dtype=float)

    # Store the result of tile_map into a new tile variable
    result_tile = wp.tile_map(wp.add, a, b)

    # Store the result_tile into the output array
    wp.tile_store(out_arr, result_tile)


res = wp.zeros(10, dtype=wp.float32, device="cuda")

wp.launch_tiled(compute, dim=[1], inputs=[res], block_dim=16)

res_t = wp.to_torch(res)
print(res_t)

tensor([1.0000, 1.1000, 1.2000, 1.3000, 1.4000, 1.5000, 1.6000, 1.7000, 1.8000,
        1.9000], device='cuda:0')


In [ ]:
import warp as wp

@wp.func
def tile_element_sqrt(x: wp.float32):
    return wp.sqrt(x)

@wp.kernel
def compute_sqrt(out_arr: wp.array(dtype=wp.float32)):

    a = wp.tile_arange(0.0, 1.0, 0.1, dtype=wp.float32)

    # Store the result of tile_map into a new tile variable
    result_tile = wp.tile_map(tile_element_sqrt, a)

    # Store the result_tile into the output array
    wp.tile_store(out_arr, result_tile)


res = wp.zeros(10, dtype=wp.float32, device="cuda")

wp.launch_tiled(compute_sqrt, dim=[1], inputs=[res], block_dim=16)

res_t = wp.to_torch(res)
print(res_t)

Module __main__ 99a3439 load on device 'cuda:0' took 244.10 ms  (compiled)
tensor([0.0000, 0.3162, 0.4472, 0.5477, 0.6325, 0.7071, 0.7746, 0.8367, 0.8944,
        0.9487], device='cuda:0')


In [ ]:

@wp.kernel
def compute_sqrt(out_arr: wp.array(dtype=float)):

    a = wp.tile_arange(0.0, 1.0, 0.1, dtype=float)

    # Store the result of tile_map into a new tile variable
    result_tile = wp.tile_map(wp.sqrt, a)

    # Store the result_tile into the output array
    wp.tile_store(out_arr, result_tile)


res = wp.zeros(10, dtype=wp.float32, device="cuda")

wp.launch_tiled(compute_sqrt, dim=[1], inputs=[res], block_dim=16)

res_t = wp.to_torch(res)
print(res_t)

Module __main__ f76c4c5 load on device 'cuda:0' took 54.43 ms  (error)


Exception: CUDA kernel build failed with error code 6

# sum(A) using 1D tile

In [ ]:
import numpy as np

TILE_SIZE = wp.constant(1024)
TILE_THREADS = 1024

@wp.kernel
def compute(a: wp.array(dtype=wp.float32), b: wp.array(dtype=wp.float32), out: wp.array(dtype=wp.float32)):

    i = wp.tid()

    a_part = wp.tile_load(a, shape=(TILE_SIZE), offset=(i*TILE_SIZE))
    b_part = wp.tile_load(b, shape=(TILE_SIZE), offset=(i*TILE_SIZE))

    # Extract the scalar value from the single-element tile returned by wp.tile_sum
    out[i] = wp.tile_sum(a_part * b_part)[0]


N = 1<<20
number_tiles = N//TILE_SIZE
print(number_tiles)

a_wp = wp.array(np.random.rand(N).astype(np.float32), device="cuda")
b_wp = wp.array(np.random.rand(N).astype(np.float32), device="cuda")
out_wp = wp.zeros(number_tiles, dtype=wp.float32, device="cuda")

wp.launch_tiled(compute, dim=[number_tiles], inputs=[a_wp, b_wp, out_wp], block_dim=TILE_THREADS)
wp.synchronize()

print(out_wp)
print(wp.utils.array_sum(out_wp))

1024
[248.03748 250.44482 242.76286 ... 244.36584 263.48776 273.9042 ]
262193.78


## Unroll (slower than wp.sum)

In [ ]:
import warp as wp

UNROLL_CHUNKS = 4

@wp.kernel
def compute_forced_unroll(a: wp.array(dtype=wp.float32),
                          b: wp.array(dtype=wp.float32),
                          out: wp.array(dtype=wp.float32)):

    # Each thread handles a unique starting point
    # i is the global thread index
    tid = wp.tid()

    # Each thread processes 4 elements
    start_idx = tid * UNROLL_CHUNKS

    local_sum = 0.0
    # Forcing 4 multiplications per thread
    local_sum += a[start_idx] * b[start_idx]
    local_sum += a[start_idx + 1] * b[start_idx + 1]
    local_sum += a[start_idx + 2] * b[start_idx + 2]
    local_sum += a[start_idx + 3] * b[start_idx + 3]

    # Store partial result
    out[tid] = local_sum

# Launching with N/4 threads
total_threads = N // UNROLL_CHUNKS
wp.launch(compute_forced_unroll, dim=total_threads, inputs=[a_wp, b_wp, out_wp])
print(out_wp)
print(wp.utils.array_sum(out_wp))

[0.69653594 0.86143523 0.5975013  ... 1.5159669  0.59294814 2.1536598 ]
1011.98645


In [ ]:
import numpy as np

TILE_SIZE = wp.constant(32)
TILE_THREADS = 32

@wp.kernel
def compute(a: wp.array2d(dtype=float), b: wp.array2d(dtype=float)):

    # obtain our block index
    i = wp.tid()

    # load a row from global memory
    t = wp.tile_load(a[i], TILE_SIZE)

    # cooperatively compute the sum of the tile elements; s is a single element tile
    s = wp.tile_sum(t)

    # store s in global memory
    wp.tile_store(b[i], s)

N = 5

a_np = np.arange(N).reshape(-1, 1) * np.ones((1, 16), dtype=float)
a = wp.array(a_np, dtype=float)
b = wp.zeros((N,1), dtype=float)
print(a_np)

wp.launch_tiled(compute, dim=[a.shape[0]], inputs=[a, b], block_dim=TILE_THREADS)

print(f"b = {b[:,0]}")

[[0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.]
 [2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2.]
 [3. 3. 3. 3. 3. 3. 3. 3. 3. 3. 3. 3. 3. 3. 3. 3.]
 [4. 4. 4. 4. 4. 4. 4. 4. 4. 4. 4. 4. 4. 4. 4. 4.]]
Module __main__ dcdc379 load on device 'cuda:0' took 1020.96 ms  (compiled)
b = [ 0. 16. 32. 48. 64.]


# Sum(A) using 2D tiles

In [ ]:
import warp as wp
import numpy as np

TILE_DIM = 16
TILE_M = wp.constant(16)
TILE_N = wp.constant(16)
TILE_THREADS = 1

@wp.kernel
def tiled_partial_reduction(A: wp.array2d(dtype=float),
                             tile_sums: wp.array(dtype=float)):
    # i, j are the tile coordinates
    i, j = wp.tid()

    # Calculate global buffer index
    tiles_per_row = A.shape[1] // TILE_DIM
    tile_id = i * tiles_per_row + j

    # Load and sum (Shared Memory/Registers)
    t = wp.tile_load(A, shape=(TILE_M, TILE_N), offset=(i*TILE_M, j*TILE_N))
    local_result = wp.tile_sum(t)

    tile_sums[tile_id] = local_result[0]
    # Setup
N = 128
data_np = np.ones((N, N), dtype=np.float32)
A = wp.array2d(data_np, device="cuda")

# How many tiles do we have total?
num_tiles = (N // TILE_DIM) * (N // TILE_DIM)
# Create a buffer to hold one sum per tile
tile_sums_buffer = wp.zeros(num_tiles, dtype=float, device="cuda")

# Launch the tiled kernel
wp.launch_tiled(
    tiled_partial_reduction,
    dim=(N // TILE_DIM, N // TILE_DIM),
    inputs=[A, tile_sums_buffer],
    block_dim=TILE_DIM * TILE_DIM # Use all threads to load/sum the tile
)
wp.synchronize()

print(f"First tile sum: {tile_sums_buffer.numpy()[1]:.2f}")
final_total = wp.utils.array_sum(tile_sums_buffer)


print(f"Final Tiled Sum: {final_total}")
print(f"Final Tiled Sum: {np.sum(data_np)}")

Module __main__ 4b74708 load on device 'cuda:0' took 878.03 ms  (compiled)
First tile sum: 256.00
Final Tiled Sum: 16384.0
Final Tiled Sum: 16384.0


# sum(A): 2D tile

In [ ]:
import numpy as np

TILE_M = wp.constant(4)
TILE_N = wp.constant(4)
TILE_THREADS = 1

@wp.kernel
def compute(a: wp.array2d(dtype=float), b: wp.array2d(dtype=float)): # Changed to wp.array2d

    i, j = wp.tid()

    t = wp.tile_load(a, shape=(TILE_M, TILE_N), offset=(i*TILE_M, j*TILE_N))

    s = wp.tile_sum(t)
    b[i,j] = s[0]

M = 8
N = 16

a_np = np.arange(M).reshape(-1, 1) * np.ones((1, N), dtype=float)
a = wp.array(a_np, dtype=float)
b = wp.zeros((M//TILE_M, N//TILE_N), dtype=float) # b is a single-element array to store the total sum

wp.launch_tiled(compute, dim=[M//TILE_M, N//TILE_N], inputs=[a, b], block_dim=TILE_THREADS)
wp.synchronize()

print(f"First tile sum: {b.numpy()[1,1]:.2f}")
final_total = wp.utils.array_sum(b)


print(f"Final Tiled Sum: {final_total}")
print(f"Final Tiled Sum: {np.sum(a_np)}")

Module __main__ a718d86 load on device 'cuda:0' took 3131.81 ms  (compiled)
First tile sum: 88.00
Final Tiled Sum: 448.0
Final Tiled Sum: 448.0


# mat mult

In [ ]:
import warp as wp
import numpy as np

# Setup
M, K, N = 1024, 1024, 1024
device = "cuda"

# Initialize
A = wp.array(np.random.randn(M, K).astype(np.float32), device=device)
B = wp.array(np.random.randn(K, N).astype(np.float32), device=device)

# The "Sovereign" One-Liner
# Note: In Warp, you often use wp.matmul for explicit library calls
C = wp.matmul(A, B)

wp.synchronize()

In [ ]:
import warp as wp

TILE_M = wp.constant(16)
TILE_N = wp.constant(16)
TILE_K = wp.constant(32)

@wp.kernel
def gemm_tiled(A: wp.array2d(dtype=float),
               B: wp.array2d(dtype=float),
               C: wp.array2d(dtype=float)):

    i, j = wp.tid()

    # allocate output tile
    sum = wp.tile_zeros(shape=(TILE_M, TILE_N), dtype=float)
    count = int(K / TILE_K)

    # iterate over inner dimension
    for k in range(count):
        a = wp.tile_load(
            A, shape=(TILE_M, TILE_K), offset=(i * TILE_M, k * TILE_K)
        )

        b = wp.tile_load(
            B, shape=(TILE_K, TILE_N), offset=(k * TILE_K, j * TILE_N)
        )


        # perform gemm + accumulate
        wp.tile_matmul(a, b, sum)

    # store result
    wp.tile_store(C, sum, offset=(i * TILE_M, j * TILE_N))


# test with 1024^2 inputs
M, N, K = 128, 128, 64

A = wp.ones((M, K), dtype=float)
B = wp.ones((K, N), dtype=float)
C = wp.empty((M, N), dtype=float)

wp.launch_tiled(gemm_tiled,
                dim=(int(M//TILE_M), int(N//TILE_N)),
                inputs=[A, B, C],
                block_dim=64)

#print(C[0,:])


Module __main__ e0cc096 load on device 'cuda:0' took 28414.43 ms  (compiled)


In [ ]:
import warp as wp
import numpy as np

wp.init()

# 1. Configuration (Tensor Cores love multiples of 16)
N = 1024
TILE_DIM = 16  # Matches the hardware fragment size
DEVICE = "cuda"

@wp.kernel
def matmul_tensor_cores(A: wp.array2d(dtype=wp.float16),
                        B: wp.array2d(dtype=wp.float16),
                        C: wp.array2d(dtype=wp.float32)):

    # Each Tiled-C thread handles one logical tile
    i, j = wp.tid()

    # Accumulator (kept in FP32 for precision)
    acc = wp.tile_zeros(shape=(TILE_DIM, TILE_DIM), dtype=wp.float32)

    # Sliding Window over K-dimension
    for k in range(N // TILE_DIM):

        # Collaborative Load from Global Memory
        # Warp 1.12.1 uses these to stage data for Tensor Cores
        t_a = wp.tile_load(A, shape=(TILE_DIM, TILE_DIM), offset=(i * TILE_DIM, k * TILE_DIM))
        t_b = wp.tile_load(B, shape=(TILE_DIM, TILE_DIM), offset=(k * TILE_DIM, j * TILE_DIM))

        # --- THE TENSOR CORE TRIGGER ---
        # wp.tile_matmul automatically maps to 'mma.sync' PTX instructions
        # when called on float16 arrays.
        wp.tile_matmul(t_a, t_b, acc)

    # Store the final FP32 result
    wp.tile_store(C, acc, offset=(i * TILE_DIM, j * TILE_DIM))

# --- Setup Manifolds ---
# Input must be float16 to engage the fastest Tensor Core paths
a_np = np.random.randn(N, N).astype(np.float16)
b_np = np.random.randn(N, N).astype(np.float16)

A_wp = wp.array(a_np, device=DEVICE)
B_wp = wp.array(b_np, device=DEVICE)
C_wp = wp.zeros((N, N), dtype=wp.float32, device=DEVICE)

# --- Launch with Tiled-C Logic ---
# dim = Total output tiles (64 x 64 = 4096 tiles total)
wp.launch_tiled(
    kernel=matmul_tensor_cores,
    dim=[N // TILE_DIM, N // TILE_DIM],
    inputs=[A_wp, B_wp, C_wp],
    block_dim=32, # Use 32 threads (1 Warp) per tile for direct MMA mapping
    device=DEVICE
)

wp.synchronize()
print(f"Tensor Core Computation Complete: {C_wp.numpy().shape}")

Module __main__ b0a947c load on device 'cuda:0' took 33377.19 ms  (compiled)
Tensor Core Computation Complete: (1024, 1024)


# derivative

In [ ]:
import warp as wp
import numpy as np

# --- KERNEL DEFINITION ---
@wp.kernel
def compute_derivative_shuffle(
    u: wp.array(dtype=wp.float32),
    du_dx: wp.array(dtype=wp.float32),
    dx: float
):
    tid = wp.tid()
    lane = tid % 32
    v_self = u[tid]

    # Register-to-register exchange (The Tile Shuffle)
    v_left = wp.shfl_index(v_self, lane - 1)
    v_right = wp.shfl_index(v_self, lane + 1)

    # Boundary/Halo handling for 32-thread warp edges
    if lane == 0:
        v_left = u[tid - 1] if tid > 0 else v_self
    if lane == 31:
        v_right = u[tid + 1] if tid < (u.size - 1) else v_self

    du_dx[tid] = (v_right - v_left) / (2.0 * dx)

# --- MAIN EXECUTION ---
if __name__ == "__main__":
    # 1. Initialize Warp
    wp.init()
    device = "cuda" # Targeted for your RTX 6000 Blackwell

    # 2. Setup Simulation Domain (Example: 1D slice of $256^3$ manifold)
    n = 1024
    L = 2.0 * np.pi
    dx = float(L / (n - 1))
    x_np = np.linspace(0.0, L, n).astype(np.float32)

    # Define a simple wave: u = sin(x), so du/dx should be cos(x)
    u_np = np.sin(x_np).astype(np.float32)

    # 3. Transfer to GPU Manifold
    u_wp = wp.from_numpy(u_np, device=device)
    du_dx_wp = wp.zeros(n, dtype=wp.float32, device=device)

    # 4. Launch Kernel
    # We use a block_dim of 128 (4 warps) as discussed for ideal SM occupancy
    wp.launch(
        kernel=compute_derivative_shuffle,
        dim=n,
        inputs=[u_wp, du_dx_wp, dx],
        block_dim=128,
        device=device
    )

    # 5. Synchronize and Retrieve
    wp.synchronize()
    du_dx_np = du_dx_wp.to_numpy()

    # 6. Verification (The Physics Check)
    expected = np.cos(x_np)
    error = np.abs(du_dx_np - expected).max()

    print(f"--- Vayusoft Performance Report ---")
    print(f"Grid Size: {n}")
    print(f"Max Analytical Error: {error:.6e}")
    print(f"Sample Output (du/dx): {du_dx_np[:5]}")

Module __main__ 57ccc72 load on device 'cuda:0' took 1.42 ms  (error)


WarpCodegenAttributeError: Error while parsing function "compute_derivative_shuffle" at /tmp/ipykernel_4308/1224761010.py:16:
    v_left = wp.shfl_index(v_self, lane - 1)
;Could not find function wp.shfl_index as a built-in or user-defined function. Note that user functions must be annotated with a @wp.func decorator to be called from a kernel.